This notebook is used to generate plots added in appendix with eval_results.csv as datasource.

# Generate Line Plot for Each Dataset

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('eval_results_dim32.csv')

variant_mapping = {
    'runA-truncate': 'MNRL + truncate',
    'runA-pca': 'MNRL + PCA',
    'runB-truncate': 'MRL + truncate',
    'nomic-truncate': 'Nomic Embed v1.5'
}

styles = {
    'MNRL + truncate': {'color': '#888888', 'marker': 's', 'linewidth': 2, 'markersize': 8},
    'MNRL + PCA': {'color': '#1f77b4', 'marker': '^', 'linewidth': 2, 'markersize': 8},
    'MRL + truncate': {'color': '#d62728', 'marker': 'o', 'linewidth': 2, 'markersize': 8},
    'Nomic Embed v1.5': {'color': '#2ca02c', 'marker': 'D', 'linewidth': 2, 'markersize': 8}
}

dimensions = [32, 64, 128, 256, 512, 768]
dim_to_idx = {dim: idx for idx, dim in enumerate(dimensions)}

datasets = df['dataset'].unique()

for dataset in datasets:
    # Filter and sort data for the current dataset
    df_ds = df[df['dataset'] == dataset].copy()
    df_ds['dim_idx'] = df_ds['dim'].map(dim_to_idx)
    df_ds = df_ds.sort_values('dim_idx')

    # Initialize plot using subplots (complying with strict figure guidelines)
    fig, ax = plt.subplots(figsize=(7, 5), dpi=300)

    # Plot each variant line
    for var_code, var_name in variant_mapping.items():
        df_var = df_ds[df_ds['variant'] == var_code]

        if not df_var.empty:
            ax.plot(
                df_var['dim_idx'],
                df_var['ndcg_at_10'], # Swap with 'recall_at_100' or 'map_at_10' if needed
                label=var_name,
                **styles[var_name]
            )

    ax.set_xlabel('Embedding dimension', fontsize=12, labelpad=10)
    ax.set_ylabel('Avg nDCG@10' if len(datasets) > 5 else 'nDCG@10', fontsize=12, labelpad=10)

    clean_title = dataset.upper() if dataset != 'nfcorpus' else 'NFCorpus'
    ax.set_title(f'Retrieval Performance on {clean_title}', fontsize=13, pad=15, fontweight='bold')

    ax.set_xticks(range(len(dimensions)))
    ax.set_xticklabels([str(d) for d in dimensions])

    ax.grid(True, linestyle='-', linewidth=0.5, color='#e5e5e5')
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.2)

    ax.legend(
        loc='lower right',
        frameon=True,
        edgecolor='black',
        framealpha=1,
        facecolor='white',
        fontsize=10
    )

    plt.tight_layout()
    plt.savefig(f'plot_{dataset}_ndcg_dim32.pdf', bbox_inches='tight')
    plt.close(fig)

print("Individual dataset plots generated successfully!")

Individual dataset plots generated successfully!


# Comparing Dim64 and Dim32 by Averaging All Datasets

## Generate Line Plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_mrl_32 = pd.read_csv('eval_results_dim32.csv') # Optimized down to 32
df_mrl_64 = pd.read_csv('eval_results_dim64.csv') # Optimized down to 64

# Compute the average across all 5 datasets for each dimension
avg_32 = df_mrl_32[df_mrl_32['variant'] == 'runB-truncate'].groupby('dim')['ndcg_at_10'].mean().reset_index()
avg_64 = df_mrl_64[df_mrl_64['variant'] == 'runB-truncate'].groupby('dim')['ndcg_at_10'].mean().reset_index()

dimensions = [32, 64, 128, 256, 512, 768]
dim_to_idx = {dim: idx for idx, dim in enumerate(dimensions)}

avg_32['dim_idx'] = avg_32['dim'].map(dim_to_idx)
avg_64['dim_idx'] = avg_64['dim'].map(dim_to_idx)

avg_32 = avg_32.sort_values('dim_idx')
avg_64 = avg_64.sort_values('dim_idx')

fig, ax = plt.subplots(figsize=(6.5, 4.5), dpi=300)

ax.plot(
    avg_32['dim_idx'], avg_32['ndcg_at_10'],
    label='MRL (Optimized down to 32)',
    color='#d62728', marker='o', linewidth=2, markersize=8
)

ax.plot(
    avg_64['dim_idx'], avg_64['ndcg_at_10'],
    label='MRL (Optimized down to 64)',
    color='#e76f51', marker='o', markerfacecolor='white',
    linestyle='--', linewidth=2, markersize=8
)

ax.set_xlabel('Embedding dimension', fontsize=11, labelpad=8)
ax.set_ylabel('Avg nDCG@10', fontsize=11, labelpad=8)

ax.set_xticks(range(len(dimensions)))
ax.set_xticklabels([str(d) for d in dimensions])

ax.grid(True, linestyle='-', linewidth=0.5, color='#e5e5e5')
ax.set_facecolor('white')
for spine in ax.spines.values():
    spine.set_color('black')
    spine.set_linewidth(1.1)

ax.legend(loc='lower right', frameon=True, edgecolor='black', framealpha=1, facecolor='white', fontsize=9.5)

plt.tight_layout()
plt.savefig('ablation_mrl_lower_bound_no-title.pdf', bbox_inches='tight')
plt.close(fig)

## Generate Table

In [4]:
import pandas as pd

file_mrl64 = 'eval_results_dim64.csv'  # Terminated at dim 64
file_mrl32 = 'eval_results_dim32.csv'  # Terminated at dim 32

df_64 = pd.read_csv(file_mrl64)
df_32 = pd.read_csv(file_mrl32)

metric = 'ndcg_at_10'

avg_64 = df_64[df_64['variant'] == 'runB-truncate'].groupby('dim')[metric].mean()
avg_32 = df_32[df_32['variant'] == 'runB-truncate'].groupby('dim')[metric].mean()

dimensions = [32, 64, 128, 256, 512, 768]
avg_64 = avg_64.reindex(dimensions)
avg_32 = avg_32.reindex(dimensions)

delta = avg_32 - avg_64

markdown_rows = []
latex_rows = []

markdown_rows.append(["Optimized down to 64"] + [f"{val:.4f}" for val in avg_64])
markdown_rows.append(
    ["Optimized down to 32"] +
    [f"**{v32:.4f}**" if v32 > v64 else f"{v32:.4f}" for v64, v32 in zip(avg_64, avg_32)]
)
markdown_rows.append(["Marginal Gain ($\Delta$)"] + [f"+{d:.4f}" if d > 0 else f"{d:.4f}" for d in delta])

latex_rows.append(["Optimized down to 64"] + [f"{val:.4f}" for val in avg_64])
latex_rows.append(
    ["Optimized down to 32"] +
    [f"\\textbf{{{v32:.4f}}}" if v32 > v64 else f"{v32:.4f}" for v64, v32 in zip(avg_64, avg_32)]
)
latex_rows.append(["\\textit{{Marginal Gain ($\Delta$)}}"] + [f"+{d:.4f}" if d > 0 else f"{d:.4f}" for d in delta])

columns = [f"d={dim}" for dim in dimensions]

df_md = pd.DataFrame(markdown_rows, columns=["Minimum Fine-Tuning Granularity"] + columns)
df_ltx = pd.DataFrame(latex_rows, columns=["Minimum Fine-Tuning Granularity"] + columns)

print("=== MARKDOWN FORMAT ===")
print(df_md.to_markdown(index=False))

print("\n=== RAW LATEX BOOKTABS ROWS ===")
for row in latex_rows:
    print(" & ".join(row) + " \\\\")

=== MARKDOWN FORMAT ===
| Minimum Fine-Tuning Granularity   | d=32       | d=64       | d=128      | d=256      |   d=512 |   d=768 |
|:----------------------------------|:-----------|:-----------|:-----------|:-----------|--------:|--------:|
| Optimized down to 64              | 0.2422     | 0.3467     | 0.3960     | 0.4315     |  0.4463 |  0.4612 |
| Optimized down to 32              | **0.2754** | **0.3609** | **0.4032** | **0.4333** |  0.4458 |  0.4582 |
| Marginal Gain ($\Delta$)          | +0.0332    | +0.0142    | +0.0072    | +0.0019    | -0.0006 | -0.003  |

=== RAW LATEX BOOKTABS ROWS ===
Optimized down to 64 & 0.2422 & 0.3467 & 0.3960 & 0.4315 & 0.4463 & 0.4612 \\
Optimized down to 32 & \textbf{0.2754} & \textbf{0.3609} & \textbf{0.4032} & \textbf{0.4333} & 0.4458 & 0.4582 \\
\textit{{Marginal Gain ($\Delta$)}} & +0.0332 & +0.0142 & +0.0072 & +0.0019 & -0.0006 & -0.0030 \\


<>:34: SyntaxWarning: invalid escape sequence '\D'
<>:42: SyntaxWarning: invalid escape sequence '\D'
<>:34: SyntaxWarning: invalid escape sequence '\D'
<>:42: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_3392/2415865894.py:34: SyntaxWarning: invalid escape sequence '\D'
  markdown_rows.append(["Marginal Gain ($\Delta$)"] + [f"+{d:.4f}" if d > 0 else f"{d:.4f}" for d in delta])
/tmp/ipykernel_3392/2415865894.py:42: SyntaxWarning: invalid escape sequence '\D'
  latex_rows.append(["\\textit{{Marginal Gain ($\Delta$)}}"] + [f"+{d:.4f}" if d > 0 else f"{d:.4f}" for d in delta])
